# 01 — Data Profiling

This notebook answers **"what exactly are we looking at"**: the
structure, completeness, uniqueness, validity, and internal
consistency of the raw data.

It deliberately does **not** look for relationships with the target
or between predictors, and it does **not** visualize anything — that
is the job of `02_eda.ipynb`. Every check here is something we can
defend as "describing the data," not "hunting for a pattern we
already suspect matters."

In [22]:
import logging
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src import config, data, profiling

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

## Load the raw files

No cleaning, no type coercion beyond what pandas infers by default —
`src/data.py` reads these exactly as they arrived.

In [23]:
train_test = data.load_train_test()
validation = data.load_validation()
val_template = data.load_validation_template()
december = data.load_december()

INFO src.data: Loading C:\dev\Spotter-ML-Engineer-Assessment\data\train_test.csv
INFO src.data: Loaded train_test.csv: 48000 rows, 14 columns
INFO src.data: Loading C:\dev\Spotter-ML-Engineer-Assessment\data\validation.csv
INFO src.data: Loaded validation.csv: 12000 rows, 13 columns
INFO src.data: Loading C:\dev\Spotter-ML-Engineer-Assessment\data\validation_predictions_template.csv
INFO src.data: Loaded validation_predictions_template.csv: 12000 rows, 2 columns
INFO src.data: Loading C:\dev\Spotter-ML-Engineer-Assessment\data\december_chart_inputs.csv
INFO src.data: Loaded december_chart_inputs.csv: 31 rows, 7 columns


## Structure: shape, columns, dtypes, a sample of rows

In [43]:
for name, df in [
    ("train_test", train_test),
    ("validation", validation),
    ("validation_template", val_template),
    ("december", december),
]:
    info = profiling.describe_structure(df, name)
    print(f"--- {info['name']} ---")
    print(f"shape: {info['n_rows']} rows x {info['n_columns']} columns")
    print(f"columns: {info['columns']}")
    print()

INFO src.profiling: Describing structure of train_test
INFO src.profiling: Describing structure of validation
INFO src.profiling: Describing structure of validation_template
INFO src.profiling: Describing structure of december


--- train_test ---
shape: 48000 rows x 14 columns
columns: ['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal', 'posted_rate']

--- validation ---
shape: 12000 rows x 13 columns
columns: ['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal']

--- validation_template ---
shape: 12000 rows x 2 columns
columns: ['load_id', 'predicted_rate']

--- december ---
shape: 31 rows x 7 columns
columns: ['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']



In [25]:
train_test.head()

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


In [26]:
train_test.dtypes

load_id          object
pickup           object
delivery         object
pickup_lat      float64
pickup_lon      float64
delivery_lat    float64
delivery_lon    float64
distance        float64
equipment        object
weight          float64
date             object
market_index    float64
quote_signal    float64
posted_rate     float64
dtype: object

## Missingness

Which columns have missing values, and how much.

In [27]:
profiling.check_missingness(train_test)

INFO src.profiling: Checking missingness across 14 columns
INFO src.profiling: Columns with missing values: 2


,n_missing,pct_missing
market_index,374,0.779
weight,300,0.625
delivery,0,0.000
pickup_lat,0,0.000
load_id,0,0.000
pickup,0,0.000
delivery_lat,0,0.000
pickup_lon,0,0.000
distance,0,0.000
delivery_lon,0,0.000


In [28]:
profiling.check_missingness(validation)

INFO src.profiling: Checking missingness across 13 columns
INFO src.profiling: Columns with missing values: 2


,n_missing,pct_missing
market_index,249,2.075
weight,165,1.375
delivery,0,0.000
pickup,0,0.000
load_id,0,0.000
pickup_lon,0,0.000
pickup_lat,0,0.000
delivery_lat,0,0.000
delivery_lon,0,0.000
equipment,0,0.000


In [29]:
profiling.check_missingness(december)

INFO src.profiling: Checking missingness across 7 columns
INFO src.profiling: Columns with missing values: 1


,n_missing,pct_missing
predicted_rate,31,100.0
delivery,0,0.0
pickup,0,0.0
distance,0,0.0
equipment,0,0.0
weight,0,0.0
date,0,0.0


## Uniqueness and duplicate rows

In [30]:
profiling.check_uniqueness(train_test, config.ID_COL)

INFO src.profiling: Checking uniqueness for id_col=load_id
INFO src.profiling: Uniqueness result: {'id_col': 'load_id', 'n_rows': 48000, 'n_unique_ids': 48000, 'is_unique': True, 'n_duplicate_ids': 0, 'n_duplicate_rows': 0}


{'id_col': 'load_id',
 'n_rows': 48000,
 'n_unique_ids': 48000,
 'is_unique': True,
 'n_duplicate_ids': 0,
 'n_duplicate_rows': 0}

In [31]:
profiling.check_uniqueness(validation, config.ID_COL)

INFO src.profiling: Checking uniqueness for id_col=load_id
INFO src.profiling: Uniqueness result: {'id_col': 'load_id', 'n_rows': 12000, 'n_unique_ids': 12000, 'is_unique': True, 'n_duplicate_ids': 0, 'n_duplicate_rows': 0}


{'id_col': 'load_id',
 'n_rows': 12000,
 'n_unique_ids': 12000,
 'is_unique': True,
 'n_duplicate_ids': 0,
 'n_duplicate_rows': 0}

## Domain / range sanity checks

Physically impossible values: non-positive distance, weight, or
posted_rate; out-of-range coordinates; pickup equal to delivery. Only
checks the columns actually present in each file.

In [32]:
profiling.check_domain_ranges(train_test)

INFO src.profiling: Running domain/range checks
INFO src.profiling: Domain/range checks complete: 8 checks run


,check,n_flagged,description
0,distance_non_positive,0,Rows where distance <= 0 (physically impossible)
1,weight_non_positive,292,Rows where weight <= 0 (physically impossible)
2,posted_rate_non_positive,0,Rows where posted_rate <= 0 (physically imposs...
3,pickup_lat_out_of_range,0,"Rows where pickup_lat is outside [-90, 90]"
4,delivery_lat_out_of_range,0,"Rows where delivery_lat is outside [-90, 90]"
5,pickup_lon_out_of_range,0,"Rows where pickup_lon is outside [-180, 180]"
6,delivery_lon_out_of_range,0,"Rows where delivery_lon is outside [-180, 180]"
7,pickup_equals_delivery,0,Rows where pickup and delivery are the same city


In [33]:
profiling.check_domain_ranges(validation)

INFO src.profiling: Running domain/range checks
INFO src.profiling: Domain/range checks complete: 7 checks run


,check,n_flagged,description
0,distance_non_positive,0,Rows where distance <= 0 (physically impossible)
1,weight_non_positive,145,Rows where weight <= 0 (physically impossible)
2,pickup_lat_out_of_range,0,"Rows where pickup_lat is outside [-90, 90]"
3,delivery_lat_out_of_range,0,"Rows where delivery_lat is outside [-90, 90]"
4,pickup_lon_out_of_range,0,"Rows where pickup_lon is outside [-180, 180]"
5,delivery_lon_out_of_range,0,"Rows where delivery_lon is outside [-180, 180]"
6,pickup_equals_delivery,0,Rows where pickup and delivery are the same city


In [34]:
profiling.check_domain_ranges(december)

INFO src.profiling: Running domain/range checks
INFO src.profiling: Domain/range checks complete: 3 checks run


,check,n_flagged,description
0,distance_non_positive,0,Rows where distance <= 0 (physically impossible)
1,weight_non_positive,0,Rows where weight <= 0 (physically impossible)
2,pickup_equals_delivery,0,Rows where pickup and delivery are the same city


## Cardinality of categorical columns

In [44]:
cardinality = profiling.check_cardinality(train_test, ["pickup", "delivery", "equipment"])
for col, info in cardinality.items():
    print(f"--- {col}: {info['n_unique']} unique values ---")
    print(info["top_values"])
    print()

INFO src.profiling: Checking cardinality for columns: ['pickup', 'delivery', 'equipment']


--- pickup: 64 unique values ---
pickup
Oklahoma City    1242
Lexington        1209
Bakersfield      1193
Fort Wayne       1170
Hartford         1150
Richmond         1140
Nashville        1124
Phoenix          1121
Baton Rouge      1115
Mobile           1094
Name: count, dtype: int64

--- delivery: 64 unique values ---
delivery
Lexington        1197
Fort Wayne       1176
Baton Rouge      1167
Bakersfield      1156
Hartford         1143
Oklahoma City    1140
Richmond         1109
Atlanta          1096
Phoenix          1090
Mobile           1089
Name: count, dtype: int64

--- equipment: 3 unique values ---
equipment
Dry Van    27202
Reefer     12045
Flatbed     8753
Name: count, dtype: int64



## Descriptive statistics for numeric columns

Plain `.describe()` output — no comment on shape or skew, no
visualization. Whether any of this needs a transform is an EDA
question, not a profiling one.

In [36]:
numeric_cols = ["distance", "weight", "market_index", "quote_signal", "posted_rate"]
profiling.describe_numeric(train_test, numeric_cols)

INFO src.profiling: Describing numeric columns: ['distance', 'weight', 'market_index', 'quote_signal', 'posted_rate']


,count,mean,std,min,25%,50%,75%,max
distance,48000.0,1135.856654,728.564416,70.00000,550.40000,953.30000,1645.525000,3439.80000
weight,47700.0,31028.844004,9391.440620,-47500.00000,25800.00000,31436.50000,37018.000000,47500.00000
market_index,47626.0,1.083387,0.168091,0.67639,0.94967,1.05580,1.219590,1.46778
quote_signal,48000.0,2.062468,0.291391,0.69228,1.89103,2.05575,2.221685,3.61035
posted_rate,48000.0,2373.980682,1486.493245,57.22000,1251.55500,2030.76000,3330.750000,25533.00000


## Distance vs. coordinates: internal consistency check

Does the given `distance` column agree with the great-circle distance
implied by the pickup/delivery coordinates? A question about whether
two columns that should structurally relate actually do — not a
question about what predicts the target.

In [37]:
distance_check = profiling.check_distance_consistency(train_test)
print("correlation between distance and great-circle distance:", distance_check["correlation"])
distance_check["ratio_describe"]

INFO src.profiling: Checking distance-vs-coordinates consistency
INFO src.profiling: Distance consistency: correlation=0.9995, 22 rows flagged (ratio > 2x)


correlation between distance and great-circle distance: 0.9995349596237125


count    48000.000000
mean         1.194681
std          0.143541
min          1.058090
25%          1.164765
50%          1.182231
75%          1.204765
max          9.634919
dtype: float64

In [38]:
print(f"{len(distance_check['flagged_rows'])} rows flagged (distance more than 2x the great-circle distance)")
distance_check["flagged_rows"][["load_id", "pickup", "delivery", "distance", "great_circle_distance", "ratio"]]

22 rows flagged (distance more than 2x the great-circle distance)


,load_id,pickup,delivery,distance,great_circle_distance,ratio
1464,TR-001465,Lubbock,Austin,70.0,34.46283,2.031174
1914,TR-001915,Lubbock,Austin,70.0,34.46283,2.031174
4140,TR-004141,New Orleans,Shreveport,70.0,7.26524,9.634919
5091,TR-005092,New Orleans,Shreveport,70.0,7.26524,9.634919
6960,TR-006961,New Orleans,Shreveport,70.0,7.26524,9.634919
8589,TR-008590,Austin,Lubbock,70.0,34.46283,2.031174
9348,TR-009349,Shreveport,New Orleans,70.0,7.26524,9.634919
11311,TR-011312,Lubbock,Austin,70.0,34.46283,2.031174
11327,TR-011328,Austin,Lubbock,78.9,34.46283,2.289423
11949,TR-011950,Shreveport,New Orleans,70.0,7.26524,9.634919


## Coordinate stability per city

Is latitude/longitude a stable, deterministic function of city name,
or does it vary row to row for the same city?

In [39]:
profiling.check_coordinate_stability(train_test, "pickup", "pickup_lat", "pickup_lon")

INFO src.profiling: Checking coordinate stability for pickup
INFO src.profiling: Coordinate stability (pickup): {'skipped': False, 'n_cities_checked': 64, 'max_lat_std': 0.0, 'max_lon_std': 0.0, 'is_fully_stable': True}


{'skipped': False,
 'n_cities_checked': 64,
 'max_lat_std': 0.0,
 'max_lon_std': 0.0,
 'is_fully_stable': True}

In [40]:
profiling.check_coordinate_stability(train_test, "delivery", "delivery_lat", "delivery_lon")

INFO src.profiling: Checking coordinate stability for delivery
INFO src.profiling: Coordinate stability (delivery): {'skipped': False, 'n_cities_checked': 64, 'max_lat_std': 0.0, 'max_lon_std': 0.0, 'is_fully_stable': True}


{'skipped': False,
 'n_cities_checked': 64,
 'max_lat_std': 0.0,
 'max_lon_std': 0.0,
 'is_fully_stable': True}

## Date coverage: train_test vs. validation

What time period does each file actually cover, and how do the two
ranges relate to each other? Purely descriptive — whether that
relationship matters for how we split the data is a later decision.

In [41]:
tt_coverage = profiling.check_date_coverage(train_test, config.DATE_COL, "train_test")
val_coverage = profiling.check_date_coverage(validation, config.DATE_COL, "validation")

INFO src.profiling: Checking date coverage for train_test
INFO src.profiling: Date coverage (train_test): {'name': 'train_test', 'n_rows': 48000, 'n_unparseable': 0, 'min_date': Timestamp('2025-01-01 00:00:00'), 'max_date': Timestamp('2025-10-31 00:00:00'), 'n_unique_dates': 304}
INFO src.profiling: Checking date coverage for validation
INFO src.profiling: Date coverage (validation): {'name': 'validation', 'n_rows': 12000, 'n_unparseable': 0, 'min_date': Timestamp('2025-11-01 00:00:00'), 'max_date': Timestamp('2025-12-31 00:00:00'), 'n_unique_dates': 61}


In [42]:
pd.DataFrame([tt_coverage, val_coverage]).set_index("name")

,n_rows,n_unparseable,min_date,max_date,n_unique_dates
name,,,,,
train_test,48000,0,2025-01-01,2025-10-31,304
validation,12000,0,2025-11-01,2025-12-31,61


## Summary

